In [ ]:
import json
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from src.data.data_sampling import get_random_sample, mapping_for_embedding

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("Data analysis") \
    .master("local[40]") \
    .config("spark.driver.memory", "30g") \
    .config("spark.executor.memory", "60g") \
    .getOrCreate()

# Preprocessing

This notebook uses as input the [raw data](https://www.transportes.gob.es/ministerio/proyectos-singulares/estudios-de-movilidad-con-big-data/opendata-movilidad) downloaded from the Spanish Ministry of Transport and Mobility. For our purposes, the daily files of trips by districts were used - `mitma-movilidad-v2/estudios_basicos/por-distritos/viajes/ficheros-diarios` - and it was obtained in early May of 2025.

The aim of our analysis is to study the effect of the flood occured the night of the 29th of October of 2024 in the Valencian community, Spain. For that reason we have selected a time window that encode two months before and two months after the hazard, from September 1st to December 31st. We have also filtered the raw data and selected only the trips occurring inside the Valencian community - Alicante, Castellon and Valencia provinces. Furthermore, the original data contains `NA` values in the gender column, which represent a big portion of the trips. Instead of ignoring these data, we applied a binomial distribution among both genders.

Everything described above is detailed in this notebook. Finally, the notebook ends up saving the preprocess data for further analysis. The idea of doing that is to make the analysis more computational efficient.

In [ ]:
# Filtering the areas that correspond to the Valencian community
# The provinces code are - Alicante: 3, Castellon: 12, Valencia: 46.

poblacion = pd.read_csv('/data/zonification/poblacion.csv', delimiter='|')
SC_districts = list(poblacion[(poblacion.provincia == 3) | (poblacion.provincia == 12) | (poblacion.provincia == 46)]['distrito'].unique())

In [ ]:
###  Reading original data from Sep 01 to Dec 31 of 2024  ###
file_path = '/data/big/davidaltamirano/day_by_day_RAW_data' 
files = [f"{file_path}/202409*_Viajes_distritos.csv.gz",
         f"{file_path}/202410*_Viajes_distritos.csv.gz",
         f"{file_path}/202411*_Viajes_distritos.csv.gz",
         f"{file_path}/202412*_Viajes_distritos.csv.gz",
        ]

raw_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("delimiter", "|") \
    .csv(files)

###  Filtering the areas only inside Valencia  ###
df_to_study = raw_df.filter((F.col("origen").isin(SC_districts)) & (F.col("destino").isin(SC_districts)))

###  Aggregating the data by period, origin, destination and gender
df_to_study = df_to_study.groupBy("fecha", "periodo", "origen", "destino", "sexo").agg(F.sum("viajes").alias("total_trips"), F.sum("viajes_km").alias("total_km"))

###  Save the refined data  ###
#storage_route = f'/data/big/davidaltamirano/data_aggregation/Study_Case/'

# Save the new aggregated DataFrame in a compressed .parquet file
#df_to_study.write \
#.option("compression", "snappy") \
#.mode("overwrite") \
#.parquet(storage_route)

In [ ]:
df_NA = df_to_study.filter(F.col('sexo') == 'NA')
df_WM = df_to_study.filter(F.col('sexo') != 'NA')

df_1, df_2 = get_random_sample(df_NA, 'total_trips')

df_women = df_1.replace('NA', 'mujer', subset=['sexo'])
df_men = df_2.replace('NA', 'hombre', subset=['sexo'])

df = df_WM.unionByName(df_women).unionByName(df_men)
df_study_case = df.groupBy('fecha', 'periodo', 'origen', 'destino', 'sexo').agg(F.sum('total_trips').alias('total_trips'))

df_study_case_renamed, renamed_dictionary = mapping_for_embedding(df_study_case)

# Save the ACTUAL DataFrame - trips only inside Valecia from Sep-Dec and no NA entries for gender
storage_route = f'/data/big/davidaltamirano/SC_Valencia_comunidad/data_refined/'
df_study_case_renamed.write \
.option("compression", "snappy") \
.mode("overwrite") \
.parquet(storage_route)

with open("/data/big/davidaltamirano/SC_Valencia_comunidad/dictionary_areas_ACTUAL_df.json", "w", encoding="utf-8") as f:
    json.dump(renamed_dictionary, f, indent=4, ensure_ascii=False)